In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sarribere99/high-resolution-sar-images-dataset-hrsid")

print("Path to dataset files:", path)

100%|██████████| 5.28G/5.28G [01:08<00:00, 82.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/ResearchProject")

folders = [
    PROJECT,
    PROJECT / "dataset",
    PROJECT / "dataset" / "images",
    PROJECT / "dataset" / "labels",
    PROJECT / "models",
    PROJECT / "results",
    PROJECT / "preprocessing",
    PROJECT / "notebooks",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ ResearchProject folder created successfully!")

for folder in folders:
    print(folder)

✅ ResearchProject folder created successfully!
/content/drive/MyDrive/ResearchProject
/content/drive/MyDrive/ResearchProject/dataset
/content/drive/MyDrive/ResearchProject/dataset/images
/content/drive/MyDrive/ResearchProject/dataset/labels
/content/drive/MyDrive/ResearchProject/models
/content/drive/MyDrive/ResearchProject/results
/content/drive/MyDrive/ResearchProject/preprocessing
/content/drive/MyDrive/ResearchProject/notebooks


In [4]:
from pathlib import Path

DATASET_PATH = Path(path)

print("Dataset path:")
print(DATASET_PATH)

print("\nDataset contents:")

for item in DATASET_PATH.iterdir():
    print(item)

Dataset path:
/root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1

Dataset contents:
/root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1/inshore_offshore
/root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1/annotations
/root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1/images


In [5]:
%%writefile /content/drive/MyDrive/ResearchProject/preprocessing/coco_to_yolo.py

import os
import json
import shutil
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

PROJECT = Path("/content/drive/MyDrive/ResearchProject")

# Dataset remains in KaggleHub cache
RAW_DIR = Path(
    "/root/.cache/kagglehub/datasets/"
    "sarribere99/high-resolution-sar-images-dataset-hrsid/"
    "versions/1"
)

OUTPUT_IMAGES = PROJECT / "dataset" / "images"
OUTPUT_LABELS = PROJECT / "dataset" / "labels"


# Create output directories

OUTPUT_IMAGES.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_LABELS.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FIND COCO ANNOTATION FILE
# ============================================================

json_files = list(
    RAW_DIR.rglob("*.json")
)

if not json_files:

    raise FileNotFoundError(
        "No COCO JSON annotation file found "
        "inside downloaded dataset."
    )


print("JSON files found:")

for file in json_files:

    print("  ", file)


# ============================================================
# USE FIRST JSON FILE
# ============================================================

ANNOTATION_FILE = json_files[0]

print("\nUsing annotation file:")
print(ANNOTATION_FILE)


# ============================================================
# LOAD COCO DATA
# ============================================================

with open(
    ANNOTATION_FILE,
    "r"
) as f:

    coco = json.load(f)


print("\nCOCO dataset information")
print("------------------------")

print(
    "Images:",
    len(coco.get("images", []))
)

print(
    "Annotations:",
    len(coco.get("annotations", []))
)

print(
    "Categories:",
    len(coco.get("categories", []))
)


# ============================================================
# CATEGORY MAPPING
# ============================================================

categories = sorted(
    coco["categories"],
    key=lambda x: x["id"]
)


category_mapping = {

    category["id"]: index

    for index, category
    in enumerate(categories)

}


print("\nCategory Mapping")
print("----------------")


for category in categories:

    print(

        f"COCO ID {category['id']} "
        f"-> YOLO ID "
        f"{category_mapping[category['id']]} "
        f"-> {category['name']}"

    )


# ============================================================
# IMAGE INFORMATION
# ============================================================

images = {

    image["id"]: image

    for image in coco["images"]

}


# ============================================================
# GROUP ANNOTATIONS BY IMAGE
# ============================================================

annotations_by_image = {}


for annotation in coco["annotations"]:

    image_id = annotation["image_id"]


    if image_id not in annotations_by_image:

        annotations_by_image[image_id] = []


    annotations_by_image[
        image_id
    ].append(annotation)


# ============================================================
# FIND ALL IMAGES
# ============================================================

image_files = {}


valid_extensions = (

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".JPG",
    ".JPEG",
    ".PNG"

)


for image_path in RAW_DIR.rglob("*"):

    if image_path.is_file():

        if image_path.suffix in valid_extensions:

            image_files[
                image_path.name
            ] = image_path


print(
    "\nImages found in downloaded dataset:",
    len(image_files)
)


# ============================================================
# COCO → YOLO CONVERSION
# ============================================================

converted = 0

missing_images = 0

total_boxes = 0


for image_id, image_info in images.items():


    filename = os.path.basename(
        image_info["file_name"]
    )


    # --------------------------------------------------------
    # Find image
    # --------------------------------------------------------

    if filename not in image_files:

        print(
            "Missing image:",
            filename
        )

        missing_images += 1

        continue


    source_image = image_files[
        filename
    ]


    # --------------------------------------------------------
    # Image dimensions
    # --------------------------------------------------------

    image_width = image_info[
        "width"
    ]

    image_height = image_info[
        "height"
    ]


    # --------------------------------------------------------
    # Copy processed image to Drive
    # --------------------------------------------------------

    destination_image = (
        OUTPUT_IMAGES / filename
    )


    shutil.copy2(

        source_image,
        destination_image

    )


    # --------------------------------------------------------
    # Create YOLO label file
    # --------------------------------------------------------

    label_filename = (

        Path(filename).stem
        + ".txt"

    )


    destination_label = (

        OUTPUT_LABELS
        / label_filename

    )


    with open(

        destination_label,
        "w"

    ) as label_file:


        annotations = (

            annotations_by_image.get(
                image_id,
                []
            )

        )


        for annotation in annotations:


            category_id = (
                annotation["category_id"]
            )


            if category_id not in category_mapping:

                continue


            class_id = (
                category_mapping[
                    category_id
                ]
            )


            # ------------------------------------------------
            # COCO bounding box
            #
            # x_min
            # y_min
            # width
            # height
            # ------------------------------------------------

            (
                x_min,
                y_min,
                box_width,
                box_height
            ) = annotation["bbox"]


            # ------------------------------------------------
            # Convert to center coordinates
            # ------------------------------------------------

            center_x = (

                x_min
                + box_width / 2

            )


            center_y = (

                y_min
                + box_height / 2

            )


            # ------------------------------------------------
            # Normalize coordinates
            # ------------------------------------------------

            center_x = (
                center_x
                / image_width
            )

            center_y = (
                center_y
                / image_height
            )

            box_width = (
                box_width
                / image_width
            )

            box_height = (
                box_height
                / image_height
            )


            # ------------------------------------------------
            # Keep values within valid range
            # ------------------------------------------------

            center_x = max(
                0,
                min(1, center_x)
            )

            center_y = max(
                0,
                min(1, center_y)
            )

            box_width = max(
                0,
                min(1, box_width)
            )

            box_height = max(
                0,
                min(1, box_height)
            )


            # ------------------------------------------------
            # Write YOLO format
            # ------------------------------------------------

            label_file.write(

                f"{class_id} "
                f"{center_x:.6f} "
                f"{center_y:.6f} "
                f"{box_width:.6f} "
                f"{box_height:.6f}\n"

            )


            total_boxes += 1


    converted += 1


# ============================================================
# SUMMARY
# ============================================================

print("\n")

print("=" * 60)

print(
    "COCO → YOLO PREPROCESSING COMPLETED"
)

print("=" * 60)


print(
    "Images converted :",
    converted
)

print(
    "Images missing   :",
    missing_images
)

print(
    "Bounding boxes   :",
    total_boxes
)


print("\nOutput images:")

print(
    OUTPUT_IMAGES
)


print("\nOutput labels:")

print(
    OUTPUT_LABELS
)


print(
    "\nRaw dataset was NOT saved to Google Drive."
)

print("\nDone!")

Writing /content/drive/MyDrive/ResearchProject/preprocessing/coco_to_yolo.py


In [6]:
!python /content/drive/MyDrive/ResearchProject/preprocessing/coco_to_yolo.py

JSON files found:
   /root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1/inshore_offshore/inshore.json
   /root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1/inshore_offshore/offshore.json
   /root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1/annotations/test2017.json
   /root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1/annotations/train2017.json
   /root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1/annotations/train_test2017.json

Using annotation file:
/root/.cache/kagglehub/datasets/sarribere99/high-resolution-sar-images-dataset-hrsid/versions/1/inshore_offshore/inshore.json

COCO dataset information
------------------------
Images: 1031
Annotations: 8206
Categories: 1

Category Mapping
----------------
COCO ID 1 -> YOLO ID 0 -> ship

Images found in downloaded d

In [7]:
%%writefile /content/drive/MyDrive/ResearchProject/preprocessing/validate_dataset.py

import os
from pathlib import Path
from PIL import Image


PROJECT = Path(
    "/content/drive/MyDrive/ResearchProject"
)

IMAGE_DIR = (
    PROJECT
    / "dataset"
    / "images"
)

LABEL_DIR = (
    PROJECT
    / "dataset"
    / "labels"
)


valid_extensions = {

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".JPG",
    ".JPEG",
    ".PNG"

}


images = [

    f

    for f in IMAGE_DIR.iterdir()

    if f.is_file()
    and f.suffix in valid_extensions

]


print("=" * 60)

print(
    "DATASET VALIDATION"
)

print("=" * 60)


print(
    "Total images:",
    len(images)
)


missing_labels = []

invalid_labels = []

corrupted_images = []

total_boxes = 0


for image_path in images:


    # ========================================================
    # CHECK IMAGE
    # ========================================================

    try:

        with Image.open(
            image_path
        ) as img:

            img.verify()


    except Exception:

        corrupted_images.append(
            image_path.name
        )

        continue


    # ========================================================
    # FIND LABEL
    # ========================================================

    label_path = (

        LABEL_DIR
        / (
            image_path.stem
            + ".txt"
        )

    )


    if not label_path.exists():

        missing_labels.append(
            image_path.name
        )

        continue


    # ========================================================
    # VALIDATE YOLO LABEL
    # ========================================================

    with open(
        label_path,
        "r"
    ) as f:

        lines = f.readlines()


    for line_number, line in enumerate(
        lines,
        1
    ):


        values = (
            line.strip().split()
        )


        # YOLO format = 5 values

        if len(values) != 5:

            invalid_labels.append(

                f"{image_path.name}: "
                f"line {line_number}"

            )

            continue


        try:

            class_id = int(
                values[0]
            )

            x = float(
                values[1]
            )

            y = float(
                values[2]
            )

            width = float(
                values[3]
            )

            height = float(
                values[4]
            )


            # Coordinates must be normalized

            if not (

                0 <= x <= 1
                and
                0 <= y <= 1
                and
                0 <= width <= 1
                and
                0 <= height <= 1

            ):

                invalid_labels.append(

                    f"{image_path.name}: "
                    f"line {line_number}"

                )

                continue


            total_boxes += 1


        except ValueError:

            invalid_labels.append(

                f"{image_path.name}: "
                f"line {line_number}"

            )


# ============================================================
# RESULTS
# ============================================================

print(
    "\nImages:",
    len(images)
)

print(
    "Total bounding boxes:",
    total_boxes
)


print(
    "\nMissing labels:",
    len(missing_labels)
)

print(
    "Invalid labels:",
    len(invalid_labels)
)

print(
    "Corrupted images:",
    len(corrupted_images)
)


# ============================================================
# DETAILS
# ============================================================

if missing_labels:

    print(
        "\nMissing label examples:"
    )

    for item in missing_labels[:10]:

        print(
            " ",
            item
        )


if invalid_labels:

    print(
        "\nInvalid label examples:"
    )

    for item in invalid_labels[:10]:

        print(
            " ",
            item
        )


if corrupted_images:

    print(
        "\nCorrupted image examples:"
    )

    for item in corrupted_images[:10]:

        print(
            " ",
            item
        )


# ============================================================
# FINAL STATUS
# ============================================================

if (

    len(missing_labels) == 0
    and
    len(invalid_labels) == 0
    and
    len(corrupted_images) == 0

):

    print(
        "\n✅ DATASET VALIDATION PASSED"
    )

else:

    print(
        "\n⚠️ DATASET HAS PROBLEMS"
    )

Writing /content/drive/MyDrive/ResearchProject/preprocessing/validate_dataset.py


In [8]:
!python /content/drive/MyDrive/ResearchProject/preprocessing/validate_dataset.py

DATASET VALIDATION
Total images: 1031

Images: 1031
Total bounding boxes: 8206

Missing labels: 0
Invalid labels: 0
Corrupted images: 0

✅ DATASET VALIDATION PASSED


In [9]:
%%writefile /content/drive/MyDrive/ResearchProject/preprocessing/split_dataset.py

import random
import shutil
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

PROJECT = Path(
    "/content/drive/MyDrive/ResearchProject"
)


SOURCE_IMAGES = (
    PROJECT
    / "dataset"
    / "images"
)

SOURCE_LABELS = (
    PROJECT
    / "dataset"
    / "labels"
)


# ============================================================
# SPLIT SETTINGS
# ============================================================

TRAIN_RATIO = 0.70

VAL_RATIO = 0.20

TEST_RATIO = 0.10

RANDOM_SEED = 42


random.seed(
    RANDOM_SEED
)


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

for split in [
    "train",
    "val",
    "test"
]:

    (
        SOURCE_IMAGES / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )


    (
        SOURCE_LABELS / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# FIND IMAGES
# ============================================================

valid_extensions = {

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".JPG",
    ".JPEG",
    ".PNG"

}


image_files = [

    file

    for file in SOURCE_IMAGES.iterdir()

    if file.is_file()

    and file.suffix in valid_extensions

]


print(
    "Total images:",
    len(image_files)
)


# ============================================================
# SHUFFLE
# ============================================================

random.shuffle(
    image_files
)


total = len(
    image_files
)


train_end = int(
    total * TRAIN_RATIO
)

val_end = (
    train_end
    + int(total * VAL_RATIO)
)


train_files = (
    image_files[:train_end]
)

val_files = (
    image_files[
        train_end:val_end
    ]
)

test_files = (
    image_files[val_end:]
)


splits = {

    "train": train_files,

    "val": val_files,

    "test": test_files

}


# ============================================================
# COPY IMAGES + LABELS
# ============================================================

for split, files in splits.items():


    print(
        f"\nProcessing {split}..."
    )


    for image_file in files:


        # ----------------------------------------------------
        # Copy image
        # ----------------------------------------------------

        destination_image = (

            SOURCE_IMAGES
            / split
            / image_file.name

        )


        shutil.copy2(

            image_file,
            destination_image

        )


        # ----------------------------------------------------
        # Corresponding label
        # ----------------------------------------------------

        label_file = (

            SOURCE_LABELS
            / (
                image_file.stem
                + ".txt"
            )

        )


        if not label_file.exists():

            print(

                "WARNING: Missing label:",
                image_file.name

            )

            continue


        destination_label = (

            SOURCE_LABELS
            / split
            / label_file.name

        )


        shutil.copy2(

            label_file,
            destination_label

        )


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 60
)

print(
    "DATASET SPLIT COMPLETED"
)

print(
    "=" * 60
)


for split, files in splits.items():


    image_count = len(

        list(

            (
                SOURCE_IMAGES
                / split
            ).glob("*")

        )

    )


    label_count = len(

        list(

            (
                SOURCE_LABELS
                / split
            ).glob("*.txt")

        )

    )


    print(

        f"{split.upper():10} "
        f"Images: {image_count:4} | "
        f"Labels: {label_count:4}"

    )


print(
    "\nRandom seed:",
    RANDOM_SEED
)

Writing /content/drive/MyDrive/ResearchProject/preprocessing/split_dataset.py


In [10]:
!python /content/drive/MyDrive/ResearchProject/preprocessing/split_dataset.py

Total images: 1031

Processing train...

Processing val...

Processing test...

DATASET SPLIT COMPLETED
TRAIN      Images:  721 | Labels:  721
VAL        Images:  206 | Labels:  206
TEST       Images:  104 | Labels:  104

Random seed: 42


In [13]:
from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/ResearchProject"
)

images_dir = PROJECT / "dataset" / "images"
labels_dir = PROJECT / "dataset" / "labels"

print("Images folder:")
for item in images_dir.iterdir():
    print(" ", item.name)

print("\nLabels folder:")
for item in labels_dir.iterdir():
    print(" ", item.name)

Images folder:
  train
  val
  test

Labels folder:
  train
  val
  test
